
## **Agente Langgraph com Gravação em Store de Longterm**
### Marcio Morelli Soares - RM371246 1AIER

In [21]:
!pip install langchain-openai -q


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langgraph.checkpoint.memory import InMemorySaver
from openai import OpenAI


#dotenv_path = Path.cwd() / ".env"
#dotenv_path = Path(r"C:\code_files\python\ai_mba\multi-agents-interoperability\multi-agents-interoperability\.env")
dotenv_path =Path("../../.env")

load_dotenv(dotenv_path=dotenv_path, override=True)
api_key = os.getenv("OPENAI_API_KEY", "").strip()
 
if not api_key:
    raise RuntimeError("OPENAI_API_KEY não foi encontrado no arquivo .env.")
 
if not api_key.startswith("sk-"):
    raise RuntimeError("OPENAI_API_KEY não é válido. Certifique-se de que a chave começa com 'sk-'.")

print("Chave de API encontrada")
print(f"OPENAI_API_KEY: {api_key[:10]}...{api_key[-10:]}")  # Exibe apenas os primeiros e últimos caracteres da chave para segurança
print(dotenv_path)

Chave de API encontrada
OPENAI_API_KEY: sk-proj-CA...lLlRy4a4EA
..\..\.env


In [30]:
import os
from langchain_openai import OpenAIEmbeddings
from langgraph.graph import StateGraph, MessagesState, START

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["OPENAI_API_KEY"],
    base_url="https://api.openai.com/v1",
)

# Testa diretamente o objeto que será usado no store
vetor = embeddings.embed_query("Teste de conexão")
print("✅ Embeddings funcionando! Dimensões:", len(vetor))

✅ Embeddings funcionando! Dimensões: 1536


In [32]:
model = init_chat_model("openai:gpt-4o-mini")

def tutor(state: MessagesState):

    return {"messages": [model.invoke(state["messages"])]}

builder = StateGraph(MessagesState)

builder.add_node("tutor", tutor)

builder.add_edge(START, "tutor")

graph = builder.compile(checkpointer=InMemorySaver())

In [33]:
import uuid
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore



store = InMemoryStore(index={"embed": "openai:text-embedding-3-small", "dims": 1536})



def tutormem(state: MessagesState, config: RunnableConfig, *, store: BaseStore):

  ns = ("memories", config["configurable"]["user_id"])

  pergunta = state["messages"][-1].content

  fatos = store.search(ns, query=pergunta, limit=3)   # LER​

  perfil = "\n".join(f.value["data"] for f in fatos)

  if "lembre" in pergunta.lower():                    # ESCREVER​

    store.put(ns, str(uuid.uuid4()), {"data": pergunta})

    sys = {"role": "system", "content": f"Perfil do aluno:\n{perfil}"}

    return {"messages": [model.invoke([sys, *state["messages"]])]}


graph = builder.compile(checkpointer=InMemorySaver(), store=store)

cfg = {"configurable": {"thread_id": "t1", "user_id": "ana"}}

In [34]:
# 1. Definindo o namespace que foi utilizado para salvar os dados
namespace = [
    ("memories", "amanda"),
    ("memories", "giulia"),
]

In [35]:
# Insere um dado passando (namespace, chave_única, dicionário_com_os_dados)
#store.put(("memories", "ana"), "user_preference", {"data": "A Ana gosta de cafe sem açucar"})

store.put(
    namespace[0],  # Amanda
    "shopping_cart",
    {
        "items": [
            "Notebook MacBook Air Rosa 512 GB",
            "Mouse sem fio Apple",
            "Capa para notebook"
        ]
    }
)

store.put(
    namespace[1],  # Giulia
    "shopping_cart",
    {
        "items": [
            "Tablet iPad Rosa 512 GB",
            "Fone de ouvido Bluetooth Bose",
            "Hub USB-C"
        ]
    }
)

In [36]:
for ns in namespace:
    carrinho = store.get(ns, "shopping_cart")

    print(f"\n=== Carrinho de {ns[1].title()} ===")

    if carrinho is not None:
        for item in carrinho.value["items"]:
            print(f"- {item}")
    else:
        print("Carrinho vazio.")


=== Carrinho de Amanda ===
- Notebook MacBook Air Rosa 512 GB
- Mouse sem fio Apple
- Capa para notebook

=== Carrinho de Giulia ===
- Tablet iPad Rosa 512 GB
- Fone de ouvido Bluetooth Bose
- Hub USB-C
